# Batch runs

A batch job halves the price and completes within 24 hours instead of streaming.
This notebook drives one end to end: write the requests, submit them, wait, read
the replies back.

Nothing here is specific to the notebook. It calls the same functions
`scripts/run.py` calls, and writes the same records live generation writes, so a
reply collected this way is indistinguishable downstream from one collected any
other way.

Only OpenAI is driven from here. Anthropic and Google have their own batch
endpoints; for those, export the file and use their console.

In [ ]:
# Import the libraries
import json
import sys
import time
from pathlib import Path

import pandas as pd

In [ ]:
# Set the working directory to the project root
if Path.cwd().name == 'notebooks':
    %cd ..

sys.path.insert(0, str(Path('scripts').resolve()))

In [ ]:
# Import the pipeline. Reloading keeps a long-lived kernel from holding an old
# copy of a script that has since changed on disk.
%load_ext autoreload
%autoreload 2

import backends
import run
import settings
import utils

# The notebook calls functions that were added to the scripts alongside it, so a
# checkout with older scripts fails deep inside a cell that has already spent
# money. Checked here instead, before anything is submitted.
needs = {'run': ['write_batch', 'read_batch', 'batch_path', 'name_after_job',
                 'set_aside_replies'],
         'utils': ['api_key', 'read_lines', 'read_table', 'result_path',
                   'model_slug', 'make_directories'],
         'backends': ['USAGE', 'spent', 'record_usage'],
         'settings': ['BATCHES_DIR', 'MODELS', 'GENERATION']}
missing = [f'{name}.{attr}' for name, attrs in needs.items()
           for attr in attrs if not hasattr(globals()[name], attr)]
if missing:
    raise SystemExit('Scripts are out of date, missing: ' + ', '.join(missing)
                     + '\nCopy scripts/ from the latest package and restart the kernel.')

# A behaviour check as well as a name check. Sampling left unset reaches OpenAI
# as its own defaults, temperature 1.0 and top_p 0.98, while the other providers
# receive what the design asks for, and no missing function name reveals that.
probe = backends.build_payload('openai', 'probe', [{'role': 'user', 'content': 'x'}],
                               settings.GENERATION['max_tokens'],
                               settings.GENERATION['temperature'])
if 'temperature' not in probe or 'top_p' not in probe:
    raise SystemExit('backends.py does not send sampling parameters to OpenAI, so '
                     'this arm would run at provider defaults.\nCopy scripts/ from '
                     'the latest package and restart the kernel.')

utils.make_directories()
pd.set_option('display.max_colwidth', 70)
print('Scripts are current')

## The model

Set the model here. It has to be one of the api models in
`config/settings.yml`, since the batch body and the price both come from its
entry in the panel.

In [ ]:
MODEL = 'gpt-5.6-luna'
ENDPOINT = '/v1/responses'

# What the panel holds, and how much of each is already ingested. A model
# reading zero has nothing in results/adaptation/, which is what write_batch
# checks, so exporting it writes a full pass even if a batch for it has been
# downloaded but not yet ingested.
prompts = utils.read_table(settings.PROMPTS_PATH)
wanted = len(prompts) * settings.GENERATION['replicates']
panel = []
for name, entry in settings.MODELS.items():
    if entry['access'] != 'api':
        continue
    have = len(utils.read_lines(utils.result_path(entry['id'],
                                                  settings.ADAPTATION_DIR)))
    panel.append({'model': entry['id'], 'provider': entry['provider'],
                  'reasoning': entry.get('reasoning') or 'default',
                  '$/M in': entry['price']['input'],
                  '$/M out': entry['price']['output'],
                  'collected': f'{have:,} of {wanted:,}'})
display(pd.DataFrame(panel).set_index('model'))

spec = next(e for e in settings.MODELS.values() if e['id'] == MODEL)
print(f"Running {MODEL}, key found in .env: "
      f"{bool(utils.api_key(spec['provider']))}")

## What is already on disk

Every pass this model has, and the sampling each was run under. Two passes
with different parameters are different configurations and should not be
pooled, which is what `FRESH` below is for.

In [ ]:
# What is already on disk for this model, and what parameters each pass used
rows = []
for kind, folder in [('collected', settings.ADAPTATION_DIR),
                     ('superseded', settings.ADAPTATION_DIR.parent / 'superseded')]:
    for file in sorted(folder.glob(f'{utils.model_slug(MODEL)}*.jsonl')):
        rows.append({'where': kind, 'file': file.name,
                     'replies': len(utils.read_lines(file)), 'sampling': ''})

for file in sorted(settings.BATCHES_DIR.glob('*_requests.jsonl')):
    body = json.loads(file.read_text().splitlines()[0])['body']
    if body.get('model') != MODEL:
        continue
    rows.append({'where': 'request file', 'file': file.name,
                 'replies': sum(1 for _ in file.open()),
                 'sampling': f"temp {body.get('temperature', 'provider default')}, "
                             f"top_p {body.get('top_p', 'provider default')}, "
                             f"reasoning {body.get('reasoning', {}).get('effort', 'default')}"})

display(pd.DataFrame(rows) if rows else 'Nothing on disk for this model yet')

## Rerunning a model

Needed only when a request parameter changes and the earlier replies are no
longer comparable, such as altering the reasoning effort or the token cap.
`FRESH` asks for every prompt again rather than only what is missing, and moves
the earlier pass to `results/superseded/`, outside the directory the pipeline
reads, so the two are never mixed. Leave it false for a normal run.

In [ ]:
FRESH = False        # True only when a request parameter has changed

if FRESH:
    moved = run.set_aside_replies(MODEL)
    print(f'Earlier pass set aside at {moved}' if moved
          else 'Nothing collected yet, so nothing to set aside')
else:
    print('Normal run: only what is missing will be requested')

## Write the requests

Anything already collected for this model is skipped, so this composes with a
run that stopped part way or with a live pass you started and abandoned.

In [ ]:
path, count = run.write_batch(MODEL, endpoint=ENDPOINT)

if path is None:
    print('Nothing outstanding for this model')
else:
    print(f'{count:,} requests written to {path}')
    print()
    print(json.dumps(json.loads(path.read_text().splitlines()[0]), indent=2))

## What it should cost

The output figure is the guess. Run twenty live first if you have not, and put
the real average here, because output is almost the whole bill.

In [ ]:
OUTPUT_TOKENS = 211          # Measured on the first full batch
INPUT_TOKENS = 24            # Measured on the first full batch

if path is None:
    print('Nothing to price, this model is already collected')
else:
    price = spec['price']
    standard = (count * INPUT_TOKENS * price['input']
                + count * OUTPUT_TOKENS * price['output']) / 1e6
    print(f'{count:,} calls at {OUTPUT_TOKENS} output tokens each')
    print(f'  Standard  ${standard:,.2f}')
    print(f'  Batched   ${standard / 2:,.2f}')

## Submit

Uploads the file and creates the job. The id is written beside the requests, so
you can come back to this notebook tomorrow and pick the job up without having
kept the kernel alive.

In [ ]:
from openai import OpenAI

client = OpenAI(api_key=utils.api_key('openai'))

uploaded = client.files.create(file=open(path, 'rb'), purpose='batch')
job = client.batches.create(input_file_id=uploaded.id, endpoint=ENDPOINT,
                            completion_window='24h')

# Written first, into a directory made on the spot: a job exists now whatever
# else fails, and its identifier is the only part that cannot be recreated.
job_file = Path(f'data/batches/{utils.model_slug(MODEL)}_job.txt')
job_file.parent.mkdir(parents=True, exist_ok=True)
job_file.write_text(job.id)
print(f'Submitted {job.id}, {job.status}')

# Then name the requests after it, so they pair with the results file the
# provider returns and a set of replies can be traced to what produced it.
print(f'Requests kept at {run.name_after_job(MODEL, job.id)}')

## Or pick up a job started in the console

A batch created from the provider's web console is not written to disk here, so
the status cell below has nothing to read. This lists the recent jobs on the
account and adopts one, which writes its id where the rest of the notebook
expects it. Run this instead of the submit cell above.

In [ ]:
from openai import OpenAI

client = OpenAI(api_key=utils.api_key('openai'))

recent = client.batches.list(limit=10)
jobs = [{'id': b.id, 'status': b.status, 'endpoint': b.endpoint,
         'total': b.request_counts.total, 'completed': b.request_counts.completed,
         'failed': b.request_counts.failed,
         'created': pd.to_datetime(b.created_at, unit='s')}
        for b in recent.data]
display(pd.DataFrame(jobs))

In [ ]:
# Adopt one: paste its id here, or take the most recent
ADOPT = jobs[0]['id'] if 'jobs' in dir() and jobs else ''

if ADOPT:
    job_file = Path(f'data/batches/{utils.model_slug(MODEL)}_job.txt')
    job_file.parent.mkdir(parents=True, exist_ok=True)
    job_file.write_text(ADOPT)
    print(f'Adopted {ADOPT} for {MODEL}')
    print('The status cell below will now find it')

## Wait

Re-run this cell rather than blocking the kernel. A job can take hours, and the
id is on disk, so nothing is lost by closing the notebook and coming back.

In [ ]:
job_file = Path(f'data/batches/{utils.model_slug(MODEL)}_job.txt')

if not job_file.exists():
    print('No job submitted for this model yet, run the cell above')
else:
    job = client.batches.retrieve(job_file.read_text().strip())
    done, failed = job.request_counts.completed, job.request_counts.failed
    total = job.request_counts.total or 1
    print(f'Job {job.id}')
    print(f'{job.status.capitalize()}, {done:,} of {total:,} done, '
          f'{failed} failed ({done / total:.0%})')

## Read the replies back

Writes into `results/adaptation/`, in the same shape as every other collected
reply, and prices what actually came back rather than what was estimated.

In [ ]:
job_file = Path(f'data/batches/{utils.model_slug(MODEL)}_job.txt')
job = (client.batches.retrieve(job_file.read_text().strip())
       if job_file.exists() else None)

if job is None or job.status != 'completed':
    print(f'Nothing to read yet: {job.status if job else "no job adopted"}')
else:
    results = run.batch_path(MODEL, 'output', job.id)
    results.write_bytes(client.files.content(job.output_file_id).read())
    print(f'Downloaded {results}')

    first = json.loads(results.read_text().splitlines()[0])
    print(f"First custom_id: {first['custom_id']}")

    backends.USAGE.update(calls=0, input=0, output=0)
    read, failed, truncated, repeated = run.read_batch(MODEL, results)

    usage, cost = backends.USAGE, backends.spent(MODEL)
    print(f'\nRead {read:,} replies, {failed} failed, {truncated} truncated, '
          f'{repeated:,} already had')
    print(f'Tokens: {usage["input"]:,} input, {usage["output"]:,} output')
    print(f'Cost: ${cost:,.2f} standard, ${cost / 2:,.2f} batched')
    print(f'Output tokens a reply: {usage["output"] / max(read - failed, 1):.0f}, '
          f'against the {OUTPUT_TOKENS} assumed above')

## Check what arrived

Empty replies are the failure to watch for on a reasoning model: reasoning
tokens count against the output cap, so a reply can come back blank having
spent its whole budget thinking.

In [ ]:
collected = utils.read_lines(utils.result_path(MODEL, settings.ADAPTATION_DIR))
prompts = utils.read_table(settings.PROMPTS_PATH)

if collected.empty:
    print(f'Nothing collected for {MODEL} yet')
else:
    blank = int((collected['response'].astype(str).str.strip() == '').sum())
    errored = int((collected['error'].astype(str).str.strip() != '').sum())
    print(f'Replies: {len(collected):,}, {blank} empty, {errored} errored')
    print(f"Coverage: {collected['prompt_id'].nunique():,} of {len(prompts):,} "
          f"prompts")

    display(collected.merge(prompts[['prompt_id', 'condition', 'prompt']],
                            on='prompt_id')[['condition', 'prompt',
                                             'response']].head(5))

## Then

Judge them:

```
python scripts/evaluate.py --backend vllm --model {MODEL} --batch-size 64
```